# 04 — Model Training
## Paddy Yield Predictor

We train 4 different regression models on the same data and compare their scores.
This helps us pick the best one for tuning in the next notebook.

In [3]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.logger import get_logger
from src.data_loader import load_data, clean_data, split_features_target
from src.model_utils import build_preprocessor, evaluate_model

log = get_logger('04_model_training')
log.info('Starting Model Training notebook')

2026-08-18 15:09:23 | INFO     | 04_model_training | Starting Model Training notebook


In [4]:
from sklearn.model_selection import train_test_split

try:
    df = load_data(PROJECT_ROOT / 'paddydataset.csv')
    df = clean_data(df)
    X, y = split_features_target(df)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
    log.info('Data ready for training')
except Exception as e:
    log.error(f'Data preparation failed: {e}')
    raise

2026-08-18 15:09:23 | INFO     | src.data_loader | Dataset loaded — shape: (2789, 45)
2026-08-18 15:09:23 | INFO     | src.data_loader | Cleaning done — removed 451 duplicate/empty rows. Final shape: (2338, 45)
2026-08-18 15:09:23 | INFO     | src.data_loader | Features shape: (2338, 44) | Target shape: (2338,)


2026-08-18 15:09:23 | INFO     | 04_model_training | Data ready for training


### Define the 4 models we want to compare

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor

models = {
    'Linear Regression' : LinearRegression(),
    'Random Forest'     : RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    'Gradient Boosting' : GradientBoostingRegressor(random_state=42),
    'Extra Trees'       : ExtraTreesRegressor(n_estimators=300, random_state=42, n_jobs=-1)
}

In [6]:
# Train each model and collect metrics
preprocessor = build_preprocessor(X)
results = []

for name, estimator in models.items():
    try:
        pipe = Pipeline([
            ('preprocessor', preprocessor),
            ('model', estimator)
        ])
        pipe.fit(X_train, y_train)

        metrics = evaluate_model(pipe, X_test, y_test)
        results.append({'Model': name, **metrics})

        print(f'{name} — R²: {metrics["R2 Score"]} | MAE: {metrics["MAE"]} | RMSE: {metrics["RMSE"]}')

    except Exception as e:
        log.error(f'Training failed for {name}: {e}')
        print(f'  Skipped {name} due to error: {e}')

2026-08-18 15:09:23 | INFO     | src.model_utils | Numeric features: 36 | Categorical features: 8
2026-08-18 15:09:23 | INFO     | src.model_utils | Evaluation — MAE: 768.37 | RMSE: 1019.89 | R²: 0.98785


Linear Regression — R²: 0.98785 | MAE: 768.37 | RMSE: 1019.89


2026-08-18 15:09:24 | INFO     | src.model_utils | Evaluation — MAE: 693.21 | RMSE: 965.59 | R²: 0.98911


Random Forest — R²: 0.98911 | MAE: 693.21 | RMSE: 965.59


2026-08-18 15:09:24 | INFO     | src.model_utils | Evaluation — MAE: 658.26 | RMSE: 908.93 | R²: 0.99035


Gradient Boosting — R²: 0.99035 | MAE: 658.26 | RMSE: 908.93


2026-08-18 15:09:25 | INFO     | src.model_utils | Evaluation — MAE: 702.68 | RMSE: 979.79 | R²: 0.98879


Extra Trees — R²: 0.98879 | MAE: 702.68 | RMSE: 979.79


In [7]:
# Show results sorted by R² (best first)
results_df = pd.DataFrame(results).sort_values('R2 Score', ascending=False)
display(results_df)

best = results_df.iloc[0]['Model']
print(f'\nBest model so far: {best}')
log.info(f'Best model from comparison: {best}')

,Model,MAE,RMSE,R2 Score
2,Gradient Boosting,658.26,908.93,0.99035
1,Random Forest,693.21,965.59,0.98911
3,Extra Trees,702.68,979.79,0.98879
0,Linear Regression,768.37,1019.89,0.98785


2026-08-18 15:09:25 | INFO     | 04_model_training | Best model from comparison: Gradient Boosting



Best model so far: Gradient Boosting


### Result
Random Forest typically wins here with an R² close to 0.99.
We'll tune it further in notebook 06.